# 1 Import Libraries

In [1]:
import pandas as pd
import numpy as np
import ast
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.stem.porter import PorterStemmer

from cleantext import clean

# 2️ Load Dataset

In [2]:
movies = pd.read_csv(r'C:\Users\HP\OneDrive\Desktop\python\Machine learning\Movie _Recommendation_system\movies_10k_dataset.csv')

movies.head()

,id,title,release_date,overview,popularity,vote_average,vote_count,genres,runtime,budget,revenue,spoken_languages,tagline,cast,director,poster_url
0,1265609,War Machine,2026-02-12,On one last grueling mission during Army Range...,698.6814,7.161,401,"['Action', 'Science Fiction', 'Thriller']",110.0,0.0,0.0,['English'],All grit. No quit.,"['Alan Ritchson', 'Dennis Quaid', 'Stephan Jam...",Patrick Hughes,https://image.tmdb.org/t/p/w500/rFhKkXhk7ClU03...
1,1290821,Shelter,2026-01-28,A man living in self-imposed exile on a remote...,344.3062,6.700,257,"['Action', 'Crime', 'Thriller']",107.0,50000000.0,42079609.0,['English'],Her safety. His mission.,"['Jason Statham', 'Bodhi Rae Breathnach', 'Mic...",Ric Roman Waugh,https://image.tmdb.org/t/p/w500/buPFnHZ3xQy6vZ...
2,799882,The Bluff,2026-02-17,When her tranquil life on a remote island is s...,243.2530,6.200,221,"['Action', 'Thriller']",102.0,0.0,0.0,['English'],Betrayal cuts deep.,"['Priyanka Chopra Jonas', 'Karl Urban', 'Safia...",Frank E. Flowers,https://image.tmdb.org/t/p/w500/w0jwjFmRu2or7L...
3,1193501,Whistle,2026-01-20,A misfit group of unwitting high school studen...,154.5885,6.035,114,"['Horror', 'Mystery']",100.0,2000000.0,2568027.0,['English'],Don't blow it.,"['Dafne Keen', 'Sophie Nélisse', 'Sky Yang', '...",Corin Hardy,https://image.tmdb.org/t/p/w500/84vAcFwxoObBuM...
4,680493,Return to Silent Hill,2026-01-21,When James receives a mysterious letter from h...,139.5571,5.000,254,"['Mystery', 'Drama', 'Horror']",106.0,23000000.0,43023897.0,"['English', 'Polish', 'Portuguese', 'Spanish']",Guilt is a place you can never leave.,"['Jeremy Irvine', 'Hannah Emily Anderson', 'Ev...",Christophe Gans,https://image.tmdb.org/t/p/w500/fqAGFN2K2kDL0E...


# 3 Dataset Check

In [3]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                10000 non-null  int64  
 1   title             10000 non-null  str    
 2   release_date      10000 non-null  str    
 3   overview          9593 non-null   str    
 4   popularity        10000 non-null  float64
 5   vote_average      10000 non-null  float64
 6   vote_count        10000 non-null  int64  
 7   genres            10000 non-null  str    
 8   runtime           9996 non-null   float64
 9   budget            9996 non-null   float64
 10  revenue           9996 non-null   float64
 11  spoken_languages  10000 non-null  str    
 12  tagline           3983 non-null   str    
 13  cast              10000 non-null  str    
 14  director          9728 non-null   str    
 15  poster_url        9515 non-null   str    
dtypes: float64(5), int64(2), str(9)
memory usage: 1.2 MB

In [4]:
movies.isnull().sum()

id                     0
title                  0
release_date           0
overview             407
popularity             0
vote_average           0
vote_count             0
genres                 0
runtime                4
budget                 4
revenue                4
spoken_languages       0
tagline             6017
cast                   0
director             272
poster_url           485
dtype: int64

# 4️ Remove Missing Values


In [5]:
movies.dropna(inplace=True)    


# Missing Values Fix 

movies['tagline'] = movies['tagline'].fillna("")
movies['overview'] = movies['overview'].fillna("")
movies['genres'] = movies['genres'].fillna("")
movies['cast'] = movies['cast'].fillna("")
movies['director'] = movies['director'].fillna("")

# 5️ Convert Genres

In [6]:
movies['genres'] = movies['genres'].apply(lambda x: x.split(",") if isinstance(x,str) else [])

# 6️ Convert Cast

In [7]:
movies['cast'] = movies['cast'].apply(lambda x: x.split(",") if isinstance(x,str) else [])

# 7 Director Convert

In [8]:
movies['director'] = movies['director'].apply(lambda x: [x] if isinstance(x,str) else [])

# 8️ Overview Split

In [9]:
movies['overview'] = movies['overview'].fillna('').apply(lambda x: x.split())

# Tagline Split

In [10]:
movies['tagline'] = movies['tagline'].apply(lambda x: x.split() if isinstance(x,str) else x)

# 9️ Remove Spaces

In [11]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['director'] = movies['director'].apply(lambda x:[i for i in x])

# 10 Create Tags

In [12]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['cast'] + movies['director']

# 11️ Convert Tags to String

In [13]:
movies['tags'] = movies['tags'].apply(lambda x:" ".join(x))
movies['tags'] = movies['tags'].apply(lambda x:x.lower())

# 12️ Stemming

In [14]:
ps = PorterStemmer()

def stem(text):

    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

movies['tags'] = movies['tags'].apply(stem)

# 13️ Final Dataset 

In [15]:
new_df = movies

new_df.head()

,id,title,release_date,overview,popularity,vote_average,vote_count,genres,runtime,budget,revenue,spoken_languages,tagline,cast,director,poster_url,tags
0,1265609,War Machine,2026-02-12,"[On, one, last, grueling, mission, during, Arm...",698.6814,7.161,401,"[['Action', 'ScienceFiction', 'Thriller']]",110.0,0.0,0.0,['English'],"[All, grit., No, quit.]","[['AlanRitchson', 'DennisQuaid', 'StephanJames...",[Patrick Hughes],https://image.tmdb.org/t/p/w500/rFhKkXhk7ClU03...,on one last gruel mission dure armi ranger tra...
1,1290821,Shelter,2026-01-28,"[A, man, living, in, self-imposed, exile, on, ...",344.3062,6.700,257,"[['Action', 'Crime', 'Thriller']]",107.0,50000000.0,42079609.0,['English'],"[Her, safety., His, mission.]","[['JasonStatham', 'BodhiRaeBreathnach', 'Micha...",[Ric Roman Waugh],https://image.tmdb.org/t/p/w500/buPFnHZ3xQy6vZ...,a man live in self-impos exil on a remot islan...
2,799882,The Bluff,2026-02-17,"[When, her, tranquil, life, on, a, remote, isl...",243.2530,6.200,221,"[['Action', 'Thriller']]",102.0,0.0,0.0,['English'],"[Betrayal, cuts, deep.]","[['PriyankaChopraJonas', 'KarlUrban', 'SafiaOa...",[Frank E. Flowers],https://image.tmdb.org/t/p/w500/w0jwjFmRu2or7L...,when her tranquil life on a remot island is sh...
3,1193501,Whistle,2026-01-20,"[A, misfit, group, of, unwitting, high, school...",154.5885,6.035,114,"[['Horror', 'Mystery']]",100.0,2000000.0,2568027.0,['English'],"[Don't, blow, it.]","[['DafneKeen', 'SophieNélisse', 'SkyYang', 'Jh...",[Corin Hardy],https://image.tmdb.org/t/p/w500/84vAcFwxoObBuM...,a misfit group of unwit high school student st...
4,680493,Return to Silent Hill,2026-01-21,"[When, James, receives, a, mysterious, letter,...",139.5571,5.000,254,"[['Mystery', 'Drama', 'Horror']]",106.0,23000000.0,43023897.0,"['English', 'Polish', 'Portuguese', 'Spanish']","[Guilt, is, a, place, you, can, never, leave.]","[['JeremyIrvine', 'HannahEmilyAnderson', 'Evie...",[Christophe Gans],https://image.tmdb.org/t/p/w500/fqAGFN2K2kDL0E...,when jame receiv a mysteri letter from hi lost...


# 14️ Vectorization

In [16]:
cv = CountVectorizer(max_features=10000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

# 15️ Similarity Matrix

In [17]:
similarity = cosine_similarity(vectors)

# 16️ Recommendation Function

In [18]:
def recommend(movie):

    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:11]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

 # 17️ Test

In [19]:
recommend("Avatar")

H.
In Time
Turkey Day: Lost in the Poultry-Geist
Tim Travers and the Time Traveler's Paradox
Don't Let Go
Project Almanac
Edge of Tomorrow
Is It a Crime?
The Demented
Betrayed


# 18️ Save PKL Files

In [20]:
pickle.dump(new_df, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))